# Amazon ML Challenge 2026: Full Production Run
### End-to-End Pipeline: Train $\rightarrow$ Predict $\rightarrow$ Validate $\rightarrow$ Package Submission

**Config Locked:**
- Embedding Model: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` (Apache-2.0, ~118M params)
- Blocking: Country partitioning + Multilingual ANN ($K_{broad}=50$) + Rare token inverted index + Exact/Levenshtein $>0.90$ name backstop ($K_{final}=20$)
- Classifier: LightGBM (offline, 300 trees, balanced class weights, tuned for per-entity macro-$F_{0.5}$)

In [ ]:
# 1. Verify GPU acceleration
!nvidia-smi

In [ ]:
# 2. Clone repository or pull latest code
!git clone https://github.com/Abbas-Hussain-byte/Amazon-ML-Challenge.git
%cd Amazon-ML-Challenge
!git pull origin main

In [ ]:
# 3. Install dependencies
!pip install -r requirements.txt
!pip install rapidfuzz lightgbm sentence-transformers torch

### Dataset Setup
Ensure `dataset/train` and `dataset/test` are in place (e.g. upload zip or mount Google Drive).

In [ ]:
# Optional: Mount Google Drive if dataset is stored there
# from google.colab import drive
# drive.mount('/content/drive')
# !unzip -q /content/drive/MyDrive/dataset.zip -d ./

# Verify data files
!ls -lh dataset/train
!ls -lh dataset/test

### Step 1: Run Full Training
Processes `dataset/train`, computes multilingual embeddings, performs blocking, fits LightGBM, and tunes decision threshold for macro-$F_{0.5}$.

In [ ]:
!python src/pipeline.py train --data-dir dataset/train --out-dir output

### Step 2: Run Full Test Prediction
Generates `output/matching_results.tsv` and `output/candidate_pairs.tsv` on the full test set.

In [ ]:
!python src/pipeline.py predict --train-dir dataset/train --test-dir dataset/test --out-dir output

### Step 3: Validate Outputs
Runs the official competition submission validator against all formatting and ID existence rules.

In [ ]:
!python utils/validate_submission.py --matching output/matching_results.tsv --candidate output/candidate_pairs.tsv --test-dir dataset/test --check-ids

### Step 4: Package Official Submission Zip
Creates `<team_name>_submission.zip` matching the required structure.

In [ ]:
!python utils/package_submission.py --team-name Null_pointers --output-tsv-dir output --zip-dest-dir output

In [ ]:
# Download final files directly from Colab
from google.colab import files
# 1. matching_results.tsv for immediate Unstop portal leaderboard upload
files.download('output/matching_results.tsv')
# 2. Full official submission zip for final artifact submission
files.download('output/Null_pointers_submission.zip')